# Socioeconomic and emissions data

Audience: researchers reproducing the national carbon-wealth workflow.

Prerequisites:
- A Python environment installed with `pip install -e ".[dev,spatial]"`.
- The repository's prepared data files. Large source files are described in the root README.

Learning goals:
- Locate the main input, intermediate, and output files for this component.
- Find the script that rebuilds the component.
- Inspect a small preview without changing any source file.

This notebook is a navigation and inspection guide. Reusable calculations remain in the scripts and modules; this notebook does not change data or run the full pipeline.

## Outline

1. Find the repository root.
2. Check the component files and preview one table.
3. Locate the rebuild commands and run a simple file check.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Run this notebook from inside the repository.')
ROOT

## Step 1 - Check the component files

The next cell reports whether each expected file is present and its dimensions when the file is a CSV. Missing large source files can be downloaded using the steps in the root README.

In [ ]:
paths = [
    ('GCB national emissions source', 'data/raw/economic/socioeconomics_2024/national_fossil_carbon_emissions_GCB2025.xlsx'),
    ('Harmonised 2024 covariates', 'data/derived/socioeconomic_2024_by_unit.csv'),
    ('Coverage audit', 'results/validation/socioeconomic_2024_coverage_audit.json'),
    ('Consolidated analysis table', 'results/validation/country_level_results_master.csv')
]

rows = []
for label, relative_path in paths:
    path = ROOT / relative_path
    row = {'file': label, 'path': relative_path, 'exists': path.is_file(), 'rows': None, 'columns': None}
    if path.is_file() and path.suffix.lower() == '.csv':
        frame = pd.read_csv(path)
        row['rows'], row['columns'] = frame.shape
    rows.append(row)

files = pd.DataFrame(rows)
display(files)

## Step 2 - Preview a table

This cell displays at most three rows and eight columns from the first available CSV. It is for checking structure and units; it does not interpret the estimates.

In [ ]:
csv_paths = [ROOT / relative_path for _, relative_path in paths if relative_path.endswith('.csv') and (ROOT / relative_path).is_file()]
if csv_paths:
    preview = pd.read_csv(csv_paths[0], nrows=3)
    display(preview.iloc[:, :min(8, preview.shape[1])])
else:
    print('No CSV from this component is present yet.')

## Rebuild commands

Run these commands from the repository root when the required inputs are available:

```bash
    python code/scripts/build_socioeconomic_2024.py
    python code/scripts/run_spatial_emissions_extension.py
```

The emissions series is territorial fossil and industrial-process CO2. It excludes land-use change emissions.

## Exercise and check

Choose one listed CSV, confirm that its row and column counts match the file table, then identify the script shown above that rebuilds it. A common mistake is to edit a derived CSV by hand; rerun the producing script instead.

In [ ]:
available = files.loc[files['exists']]
assert not available.empty, 'No component files were found; check the repository root and prerequisites.'
print(f"Found {len(available)} of {len(files)} listed component files.")
print('First available file:', available.iloc[0]['path'])